In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

PyTorch: 2.12.1+cpu
Pandas: 1.5.3
NumPy: 1.26.4


In [2]:
df = pd.read_hdf("metr-la.h5")

print(df.shape)
df.head()

(34272, 207)


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,63.333333,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


In [3]:
data = df.values

print(data.shape)

print("Min:", data.min())
print("Max:", data.max())
print("Mean:", data.mean())

(34272, 207)
Min: 0.0
Max: 70.0
Mean: 53.71902110241344


In [4]:
scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(data)

print(data_scaled.shape)

(34272, 207)


In [5]:
SEQ_LEN = 12

X = []
y = []

for i in range(len(data_scaled) - SEQ_LEN):
    
    X.append(
        data_scaled[i:i+SEQ_LEN]
    )
    
    y.append(
        data_scaled[i+SEQ_LEN]
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (34260, 12, 207)
y shape: (34260, 207)


In [6]:
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.1)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[
    train_size:
    train_size+val_size
]

y_val = y[
    train_size:
    train_size+val_size
]

X_test = X[
    train_size+val_size:
]

y_test = y[
    train_size+val_size:
]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(23982, 12, 207)
(3426, 12, 207)
(6852, 12, 207)


In [7]:
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)

X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val)

X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

In [8]:
batch_size = 64

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [9]:
corr = np.corrcoef(data.T)

print(corr.shape)
print(corr.min())
print(corr.max())

(207, 207)
0.1752069082484275
1.0


In [10]:
threshold = 0.8

adj = (corr > threshold).astype(np.float32)

np.fill_diagonal(adj, 0)

print(adj.shape)
print("Edges:", adj.sum())

(207, 207)
Edges: 3630.0


In [11]:
num_nodes = adj.shape[0]

density = adj.sum() / (num_nodes * (num_nodes - 1))

print("Density:", density)

Density: 0.08512733924299985


In [12]:
A = adj + np.eye(adj.shape[0])

D = np.diag(
    np.sum(A, axis=1)
)

D_inv_sqrt = np.linalg.inv(
    np.sqrt(D)
)

A_hat = D_inv_sqrt @ A @ D_inv_sqrt

print(A_hat.shape)

(207, 207)


In [13]:
threshold = 0.8

adj = (corr > threshold).astype(np.float32)

np.fill_diagonal(adj, 0)

print("Edges:", adj.sum())

density = adj.sum() / (207 * 206)

print("Density:", density)

Edges: 3630.0
Density: 0.08512733924299985


In [14]:
import torch

A_hat = torch.FloatTensor(A_hat)

In [15]:
class GCNLayer(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)

    def forward(self, X, A_hat):

        X = torch.matmul(A_hat, X)

        X = self.linear(X)

        return torch.relu(X)


class GCNLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.gcn = GCNLayer(
            in_features=12,
            out_features=32
        )

        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            207
        )

    def forward(self, x):

        # x: (batch,12,207)

        x = x.permute(0, 2, 1)

        # (batch,207,12)

        x = self.gcn(x, A_hat)

        # (batch,207,32)

        x, _ = self.lstm(x)

        # (batch,207,64)

        x = x[:, -1, :]

        # (batch,64)

        x = self.fc(x)

        # (batch,207)

        return x

In [16]:
model = GCNLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {avg_loss:.6f}"
    )

Epoch 1/30 Loss: 0.074940
Epoch 2/30 Loss: 0.038758
Epoch 3/30 Loss: 0.036299
Epoch 4/30 Loss: 0.035583
Epoch 5/30 Loss: 0.034717
Epoch 6/30 Loss: 0.033475
Epoch 7/30 Loss: 0.032073
Epoch 8/30 Loss: 0.028920
Epoch 9/30 Loss: 0.027771
Epoch 10/30 Loss: 0.027077
Epoch 11/30 Loss: 0.026747
Epoch 12/30 Loss: 0.026401
Epoch 13/30 Loss: 0.026067
Epoch 14/30 Loss: 0.025743
Epoch 15/30 Loss: 0.025343
Epoch 16/30 Loss: 0.025146
Epoch 17/30 Loss: 0.024871
Epoch 18/30 Loss: 0.024597
Epoch 19/30 Loss: 0.024347
Epoch 20/30 Loss: 0.024293
Epoch 21/30 Loss: 0.024277
Epoch 22/30 Loss: 0.024007
Epoch 23/30 Loss: 0.023901
Epoch 24/30 Loss: 0.023412
Epoch 25/30 Loss: 0.023105
Epoch 26/30 Loss: 0.022808
Epoch 27/30 Loss: 0.022492
Epoch 28/30 Loss: 0.022250
Epoch 29/30 Loss: 0.021996
Epoch 30/30 Loss: 0.021877


In [18]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

model.eval()

with torch.no_grad():
    predictions = model(X_test)

predictions = predictions.numpy()
true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.09746249
RMSE: 0.17655063
